In [2]:
import pandas as pd
import re
import nltk

In [6]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [7]:
df = pd.read_csv('../Dataset/IMDB Dataset.csv')
df = df.head(20000)

In [9]:
stop_word = set(stopwords.words('english'))

In [8]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [12]:
def preprocessing(text):
    text = re.sub(r'<.*?>','',text)
    text = re.sub(r'[^a-z\s]','',text)
    text = re.sub(r's\+',' ',text)
    text = re.sub(r'http\S+','',text)
    text = text.strip()

    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(token,pos ='v') for token in tokens if token not in stop_word]

    text = ' '.join(tokens)
    return text

In [13]:
df['review'] = df['review'].map(preprocessing)

In [15]:
voc_size = 150

from tensorflow.keras.preprocessing.text import one_hot

one_hot_repr = [one_hot(sent, voc_size) for sent in df['review']]


In [19]:
maxLen = 200

from tensorflow.keras.preprocessing.sequence import pad_sequences

x = pad_sequences(one_hot_repr, maxlen = maxLen, padding = 'post', dtype = 'float32')

In [21]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [22]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state=42)

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Embedding

In [25]:
model = Sequential()

model.add(Embedding(voc_size, 128, input_shape = (maxLen, )))
model.add(SimpleRNN(128 , activation='relu'))
model.add(Dense(1, activation = 'sigmoid'))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │        19,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,225 (204.00 KB)

 Trainable params: 52,225 (204.00 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics =['accuracy'])

In [27]:
history = model.fit(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 36s 70ms/step - accuracy: 0.5008 - loss: 0.6934 - val_accuracy: 0.5242 - val_loss: 0.6923
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 65ms/step - accuracy: 0.5024 - loss: 0.6930 - val_accuracy: 0.4900 - val_loss: 0.6926
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 2494s 5s/step - accuracy: 0.5073 - loss: 0.6925 - val_accuracy: 0.4873 - val_loss: 0.6926
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 669s 1s/step - accuracy: 0.5113 - loss: 0.6919 - val_accuracy: 0.5185 - val_loss: 0.6927
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 67ms/step - accuracy: 0.5081 - loss: 0.6914 - val_accuracy: 0.4865 - val_loss: 0.6938


In [28]:
y_pred = model.predict(x_test)

125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step


In [29]:
y_pred = y_pred.argmax(axis =1)

In [30]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
accuracy

0.512